In [0]:
# ==========================================
# STEP 2: SILVER LAYER - Data Quality (Kaggle Edition)
# ==========================================
from pyspark.sql.functions import col, to_timestamp, monotonically_increasing_id

bronze_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/bronze/"
silver_clean_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/silver/clean_events/"

df_bronze = spark.read.format("delta").load(bronze_s3_path)

print("Applying Data Quality gates to Kaggle Data...")
df_clean = df_bronze \
    .filter(col("user_id").isNotNull()) \
    .filter(col("event_time").isNotNull()) \
    .withColumn("event_time", to_timestamp(col("event_time"))) \
    .withColumn("user_id", col("user_id").cast("integer")) \
    .withColumn("event_id", monotonically_increasing_id()) # Safely generate a unique ID for every click

df_clean.write.format("delta").mode("overwrite").save(silver_clean_s3_path)
print("✅ Silver cleaning complete!")

In [0]:
# ==========================================
# STEP 3: SILVER LAYER - Sessionization
# ==========================================
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, unix_timestamp, when, sum as spark_sum, concat_ws

# 1. Define Paths
silver_clean_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/silver/clean_events/"
silver_sessions_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/silver/sessions/"

print("Loading clean data...")
df_clean = spark.read.format("delta").load(silver_clean_s3_path)

# 2. Define the Spark Window
# This tells Spark to group data by user and sort it by time
user_window = Window.partitionBy("user_id").orderBy("event_time")

print("Calculating time gaps between events...")
# 3. Find the timestamp of the previous event for each user
df_with_lag = df_clean.withColumn("prev_event_time", lag("event_time").over(user_window))

# 4. Calculate the gap in seconds between current and previous event
df_with_gap = df_with_lag.withColumn(
    "gap_seconds",
    unix_timestamp(col("event_time")) - unix_timestamp(col("prev_event_time"))
)

print("Identifying session boundaries (30-minute timeout)...")
# 5. Flag a row as '1' if it is a new session (gap >= 1800 seconds or first event)
df_new_session_flag = df_with_gap.withColumn(
    "is_new_session",
    when(col("prev_event_time").isNull(), 1)
    .when(col("gap_seconds") >= 1800, 1)
    .otherwise(0)
)

# 6. Create a rolling session number for the user using a cumulative sum
df_session_numbered = df_new_session_flag.withColumn(
    "user_session_num",
    spark_sum("is_new_session").over(user_window)
)

# 7. Create a globally unique session ID (e.g., "12345_1", "12345_2")
df_final_sessions = df_session_numbered.withColumn(
    "session_id",
    concat_ws("_", col("user_id"), col("user_session_num"))
).drop("prev_event_time", "is_new_session", "user_session_num") 
# We drop the temp columns to keep it clean, but we keep 'gap_seconds' for the ML model later!

print(f"Writing sessionized data to {silver_sessions_s3_path}...")
df_final_sessions.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_sessions_s3_path)

print(" Sessionization complete! Stateless clicks are now stateful sessions.")


# Find the real users with the most events in the Kaggle data
print("--- Top 5 Most Active Real Users ---")
display(
    df_final_sessions.groupBy("user_id").count().orderBy(col("count").desc()).limit(5)
)